# Landmark measurement model: notebook

Three coloured poles stand at known places in the 5 m x 4 m room. The robot at pose A = (2, 2, 0) reads the range and bearing of each. We compute expected readings, the density of the readings, where the robot can be, and we draw poses from one reading.

In [1]:
import numpy as np

LM = {"pink": (4.4, 3.8), "green": (0.4, 3.2), "blue": (3.2, 0.4)}
POSE_A = (2.0, 2.0, 0.0)
SIG_R = 0.1
SIG_PHI = np.radians(5)
READ = {"pink": (3.10, np.radians(40)), "green": (1.95, np.radians(141)), "blue": (2.05, np.radians(-55))}


def wrap(a):
    """Angle wrapped to (-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def expected(pose, m):
    x, y, th = pose
    dx, dy = m[0] - x, m[1] - y
    return np.hypot(dx, dy), wrap(np.arctan2(dy, dx) - th)


def gauss(v, sig):
    return np.exp(-0.5 * (v / sig) ** 2) / (sig * np.sqrt(2 * np.pi))


def landmark_density(pose, name, reading=None):
    r, phi = reading or READ[name]
    rh, ph = expected(pose, LM[name])
    return gauss(r - rh, SIG_R) * gauss(wrap(phi - ph), SIG_PHI)


def range_density_xy(X, Y, name):
    """Range part only, over positions (x, y): high on a ring of radius r around the landmark."""
    m = LM[name]
    return gauss(READ[name][0] - np.hypot(m[0] - X, m[1] - Y), SIG_R)


def sample_poses(name, n, rng):
    """Poses consistent with one reading of landmark `name` (Thrun Table 6.5)."""
    m = LM[name]
    r, phi = READ[name]
    g = rng.uniform(0, 2 * np.pi, n)
    rr = r + rng.normal(0, SIG_R, n)
    pp = phi + rng.normal(0, SIG_PHI, n)
    return m[0] + rr * np.cos(g), m[1] + rr * np.sin(g), wrap(g + np.pi - pp)


## 1. Expected range and bearing, and the density of each reading (Note, Sections 3 and 4)

In [2]:
import pandas as pd
rows = []
for n in LM:
    rh, ph = expected(POSE_A, LM[n])
    r, phi = READ[n]
    rows.append(dict(pole=n, r_hat=round(rh, 3), phi_hat_deg=round(np.degrees(ph), 2), r=r, phi_deg=round(np.degrees(phi), 1),
                     dr=round(r - rh, 3), dphi_deg=round(np.degrees(wrap(phi - ph)), 2),
                     p_range=round(gauss(r - rh, SIG_R), 4), p_bearing=round(gauss(wrap(phi - ph), SIG_PHI), 4),
                     density=round(landmark_density(POSE_A, n), 3)))
pd.DataFrame(rows)

,pole,r_hat,phi_hat_deg,r,phi_deg,dr,dphi_deg,p_range,p_bearing,density
0,pink,3.0,36.87,3.10,40.0,0.10,3.13,2.4197,3.7580,9.093
1,green,2.0,143.13,1.95,141.0,-0.05,-2.13,3.5207,4.1750,14.699
2,blue,2.0,-53.13,2.05,-55.0,0.05,-1.87,3.5207,4.2628,15.008


In [3]:
B = (3.0, 2.0, 0.0)
pA = np.prod([landmark_density(POSE_A, n) for n in LM])
pB = np.prod([landmark_density(B, n) for n in LM])
print('all three readings: pose A', round(pA, 1), ' pose B', pB)
for n in LM:
    rh, ph = expected(B, LM[n]); print('B', n, round(rh, 3), round(np.degrees(ph), 2), landmark_density(B, n))

all three readings: pose A 2005.9  pose B 1.350474117401158e-43
B pink 2.28 52.13 2.486103452707965e-15
B green 2.864 155.22 2.4005618091993794e-19
B blue 1.612 -82.87 2.262841712626313e-10


## 2. atan2 and angle wrapping (Note, Section 3)

In [4]:
print('arctan(1.2 / -1.6) =', np.degrees(np.arctan(1.2 / -1.6)).round(2), ' atan2(1.2, -1.6) =', np.degrees(np.arctan2(1.2, -1.6)).round(2))
print('175 - (-178) =', 175 - (-178), ' wrapped:', np.degrees(wrap(np.radians(175) - np.radians(-178))).round(2))

arctan(1.2 / -1.6) = -36.87  atan2(1.2, -1.6) = 143.13
175 - (-178) = 353  wrapped: -7.0


A check with other numbers: robot at (0, 0, 0), pole at (3, 4), reading 5.1 m and 60°, both spreads 0.5. The product is a density, not a probability.

In [5]:
rh, ph = expected((0, 0, 0), (3, 4))
print(rh, ph.round(4), gauss(5.1 - rh, 0.5).round(4), gauss(np.radians(60) - ph, 0.5).round(4),
      (gauss(5.1 - rh, 0.5) * gauss(np.radians(60) - ph, 0.5)).round(4))

5.0 0.9273 0.7821 0.7753 0.6063


## 3. Two ranges: where the circles meet (Note, Section 5.2)

In [6]:
p1, p2 = np.array(LM['pink']), np.array(LM['green'])
d1, d2 = READ['pink'][0], READ['green'][0]
a = np.linalg.norm(p2 - p1)
x = (a ** 2 + d1 ** 2 - d2 ** 2) / (2 * a)
y = np.sqrt(d1 ** 2 - x ** 2)
u = (p2 - p1) / a; nrm = np.array([-u[1], u[0]])
print('a =', a.round(4), ' x =', x.round(4), ' y = +-', y.round(4))
print('meeting points:', (p1 + x * u + y * nrm).round(3), (p1 + x * u - y * nrm).round(3))

a = 4.0447  x = 2.7403  y = +- 1.4494
meeting points: [1.905 1.96 ] [1.475 4.827]


In [7]:
xs, ys = np.linspace(0, 5, 251), np.linspace(0, 4, 201)
X, Y = np.meshgrid(xs, ys)
Z = range_density_xy(X, Y, 'pink') * range_density_xy(X, Y, 'green') * range_density_xy(X, Y, 'blue')
i, j = np.unravel_index(Z.argmax(), Z.shape)
print('highest point with all three ranges:', xs[j].round(2), ys[i].round(2))

highest point with all three ranges: 1.9 1.98


## 4. Sampling poses from one reading (Note, Section 6)

In [8]:
g = np.arctan2(2 - 3.8, 2 - 4.4)                       # direction from the pink pole to A
print('gamma =', np.degrees(g).round(2), 'deg')
for r, phi in [(3.0, np.arctan2(1.8, 2.4)), READ['pink']]:
    print('r', r, 'phi', np.degrees(phi).round(2), '->', (4.4 + r * np.cos(g)).round(3), (3.8 + r * np.sin(g)).round(3),
          np.degrees(wrap(g + np.pi - phi)).round(2))

gamma = -143.13 deg
r 3.0 phi 36.87 -> 2.0 2.0 0.0
r 3.1 phi 40.0 -> 1.92 1.94 -3.13


In [9]:
rng = np.random.default_rng(4)
x, y, th = sample_poses('pink', 200, rng)
r_back = np.hypot(4.4 - x, 3.8 - y)
phi_back = np.degrees(wrap(np.arctan2(3.8 - y, 4.4 - x) - th))
print('range seen from the samples: mean', r_back.mean().round(3), 'sd', r_back.std().round(3))
print('bearing seen from the samples: mean', phi_back.mean().round(2), 'sd', phi_back.std().round(2))
inside = (x > 0) & (x < 5) & (y > 0) & (y < 4)
print('samples inside the room:', inside.sum(), 'of', len(x))

range seen from the samples: mean 3.103 sd 0.103
bearing seen from the samples: mean 39.79 sd 4.9
samples inside the room: 64 of 200


## 5. Try it: move the robot and watch the densities

In [10]:
import plotly.graph_objects as go
fig = go.Figure()
ys_ = np.round(np.arange(1.0, 3.01, 0.25), 2)
xs_ = np.linspace(0.5, 4.5, 161)
for k, yy in enumerate(ys_):
    vals = [np.log(np.prod([landmark_density((xx, yy, 0.0), n) for n in LM])) for xx in xs_]
    fig.add_trace(go.Scatter(x=xs_, y=vals, visible=(k == 4), name=f'y = {yy}'))
steps = [dict(method='update', args=[{'visible': [j == k for j in range(len(ys_))]}], label=str(v)) for k, v in enumerate(ys_)]
fig.update_layout(sliders=[dict(steps=steps, active=4, currentvalue=dict(prefix='robot y (m): '))],
                  xaxis_title='robot x (m), heading 0', yaxis_title='log density of the three readings', yaxis_range=[-400, 10])
fig